# 05 — Cross-Network Alignment (Menaikkan MCC Cross-Network)

**Pertanyaan inti:** bisakah MCC cross-network dinaikkan dari ~0 (baseline T3) dengan menyelaraskan distribusi fitur antar-dataset?

**Tiga strategi dibandingkan (Model A, 9 fitur, biner):**
1. **Baseline (single-source)** — latih 1 dataset, uji dataset lain. Pembanding (replikasi T3).
2. **Joint training** — latih pada GABUNGAN CIC+UNSW, uji test masing-masing. Mengukur apakah fitur irisan cukup ekspresif bila kedua jaringan terlihat. (Bingkai jujur: ini *multi-source training*, bukan generalisasi ke jaringan tak-terlihat.)
3. **CORAL** — *unsupervised domain adaptation*: selaraskan statistik orde-2 (kovarians) source→target TANPA label target, lalu latih di source ter-align, uji target. Ini uji generalisasi sejati.

**Preprocessing:** z-score per dataset (Opsi 3). **Metrik:** MCC (utama), F1, ACC.

**Hipotesis jujur:**
- Joint training kemungkinan membantu (model melihat kedua distribusi) — tapi bukan bukti generalisasi ke jaringan baru.
- CORAL menguji adaptasi sejati; jika MCC cross naik = temuan kuat; jika tidak = gap terlalu dalam utk alignment orde-2 (tetap temuan berharga).
- Semua angka dilaporkan apa adanya.

> Jalankan di SageMaker (butuh `cleaned_100.pkl` + CSV UNSW di `../data/`).

In [ ]:
# --- Bootstrap dependency ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn'), ('xgboost','xgboost'), ('scipy','scipy')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from scipy.linalg import sqrtm
from xgboost import XGBClassifier

CIC_PKL    = '../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN = '../data/UNSW_NB15_testing-set.csv'   # 175.341 -> TRAIN
UNSW_TEST  = '../data/UNSW_NB15_training-set.csv'  #  82.332 -> TEST
OUT_JSON   = '../cross_network_alignment.json'
RANDOM_SEED = 42
print('CIC pkl    :', os.path.exists(CIC_PKL))
print('UNSW train :', os.path.exists(UNSW_TRAIN))
print('UNSW test  :', os.path.exists(UNSW_TEST))

In [ ]:
# --- Mapping fitur Model A ---
MAP_A = {
    'duration'  : ('Flow Duration',    'dur'),
    'fwd_pkts'  : ('Tot Fwd Pkts',     'spkts'),
    'bwd_pkts'  : ('Tot Bwd Pkts',     'dpkts'),
    'fwd_bytes' : ('TotLen Fwd Pkts',  'sbytes'),
    'bwd_bytes' : ('TotLen Bwd Pkts',  'dbytes'),
    'fwd_mean'  : ('Fwd Pkt Len Mean', 'smean'),
    'bwd_mean'  : ('Bwd Pkt Len Mean', 'dmean'),
    'src_load'  : ('Flow Byts/s',      'sload'),
    'dst_load'  : ('Bwd Pkts/s',       'dload'),
}
CANON = list(MAP_A.keys())
print('Model A:', CANON)

In [ ]:
# --- Muat CIC (un-scale) + label biner (label_mapping Benign=0) ---
with open(CIC_PKL, 'rb') as f:
    d = pickle.load(f)
cic_feats = list(d['feature_names'])
X = np.asarray(d['X'], dtype=float)
scaler = d.get('scaler', None)
X_orig = X * scaler.scale_ + scaler.mean_ if (scaler is not None and hasattr(scaler,'scale_')) else X
cic_df = pd.DataFrame(X_orig, columns=cic_feats)
y_cic_raw = np.asarray(d['y'])
lm = d.get('label_mapping', {})
benign_code = lm.get('Benign', 0)
y_cic_bin = (y_cic_raw != benign_code).astype(int)
print('CIC:', cic_df.shape, '| biner:', dict(pd.Series(y_cic_bin).value_counts()), f'| Benign={benign_code}')

In [ ]:
# --- Muat UNSW ---
unsw_tr = pd.read_csv(UNSW_TRAIN); unsw_te = pd.read_csv(UNSW_TEST)
y_unsw_tr = unsw_tr['label'].astype(int).values
y_unsw_te = unsw_te['label'].astype(int).values
print('UNSW train:', unsw_tr.shape, '| test:', unsw_te.shape)

In [ ]:
# --- Util ---
def build_matrix(df, side):
    idx = 0 if side == 'cic' else 1
    cols = [MAP_A[c][idx] for c in CANON]
    out = df[cols].copy(); out.columns = CANON
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def make_xgb():
    return XGBClassifier(objective='binary:logistic', eval_metric='logloss',
        max_depth=8, learning_rate=0.1, n_estimators=200, subsample=0.8,
        colsample_bytree=0.8, n_jobs=-1, random_state=RANDOM_SEED, tree_method='hist')

def ev(y_true, y_pred):
    return dict(mcc=float(matthews_corrcoef(y_true, y_pred)),
                f1=float(f1_score(y_true, y_pred, zero_division=0)),
                acc=float(accuracy_score(y_true, y_pred)),
                confusion=confusion_matrix(y_true, y_pred).tolist())

In [ ]:
# --- Siapkan matriks + z-score per dataset ---
Xc_all = build_matrix(cic_df, 'cic')
Xu_tr_raw = build_matrix(unsw_tr, 'unsw')
Xu_te_raw = build_matrix(unsw_te, 'unsw')

Xc_tr_raw, Xc_te_raw, yc_tr, yc_te = train_test_split(
    Xc_all, y_cic_bin, test_size=0.3, random_state=RANDOM_SEED, stratify=y_cic_bin)

sc_cic = StandardScaler().fit(Xc_tr_raw)
Xc_tr = sc_cic.transform(Xc_tr_raw); Xc_te = sc_cic.transform(Xc_te_raw)
sc_unsw = StandardScaler().fit(Xu_tr_raw)
Xu_tr = sc_unsw.transform(Xu_tr_raw); Xu_te = sc_unsw.transform(Xu_te_raw)
print('CIC train/test:', Xc_tr.shape, Xc_te.shape, '| UNSW train/test:', Xu_tr.shape, Xu_te.shape)

In [ ]:
# ============================================================
# STRATEGI 1 — BASELINE (single-source) : replikasi T3
# ============================================================
results = {}

# latih CIC -> uji UNSW
m = make_xgb(); m.fit(Xc_tr, yc_tr)
base_cic2unsw = ev(y_unsw_te, m.predict(Xu_te))
base_cic_same = ev(yc_te, m.predict(Xc_te))
# latih UNSW -> uji CIC
m = make_xgb(); m.fit(Xu_tr, y_unsw_tr)
base_unsw2cic = ev(yc_te, m.predict(Xc_te))
base_unsw_same = ev(y_unsw_te, m.predict(Xu_te))

results['baseline'] = dict(cic_same=base_cic_same, unsw_same=base_unsw_same,
                           cic2unsw=base_cic2unsw, unsw2cic=base_unsw2cic)
print('BASELINE (single-source)')
print(f'  CIC same  MCC={base_cic_same["mcc"]:.4f} | UNSW same MCC={base_unsw_same["mcc"]:.4f}')
print(f'  CIC->UNSW MCC={base_cic2unsw["mcc"]:+.4f} | UNSW->CIC MCC={base_unsw2cic["mcc"]:+.4f}')

In [ ]:
# ============================================================
# STRATEGI 2 — JOINT TRAINING (gabung CIC+UNSW pada data latih)
# ============================================================
# Catatan: fitur sudah ter-z-score PER dataset, jadi keduanya di ruang standar.
# Untuk keseimbangan, subsample CIC train agar tak mendominasi (CIC jauh lebih besar).
n_unsw = len(Xu_tr)
rng = np.random.RandomState(RANDOM_SEED)
cic_idx = rng.choice(len(Xc_tr), min(len(Xc_tr), n_unsw*3), replace=False)  # rasio CIC:UNSW ~3:1
X_joint = np.vstack([Xc_tr[cic_idx], Xu_tr])
y_joint = np.concatenate([yc_tr[cic_idx], y_unsw_tr])
print(f'Joint train: {len(X_joint):,} ({len(cic_idx):,} CIC + {n_unsw:,} UNSW)')

mj = make_xgb(); mj.fit(X_joint, y_joint)
joint_cic = ev(yc_te, mj.predict(Xc_te))
joint_unsw = ev(y_unsw_te, mj.predict(Xu_te))
results['joint'] = dict(cic_test=joint_cic, unsw_test=joint_unsw)
print('JOINT TRAINING (uji masing-masing test)')
print(f'  uji CIC  MCC={joint_cic["mcc"]:.4f} F1={joint_cic["f1"]:.4f}')
print(f'  uji UNSW MCC={joint_unsw["mcc"]:.4f} F1={joint_unsw["f1"]:.4f}')

In [ ]:
# ============================================================
# STRATEGI 3 — CORAL (CORrelation ALignment), unsupervised DA
# ============================================================
# Align kovarians source -> target. Label target TIDAK dipakai saat align.
# Rumus (Sun et al. 2016): C_s = cov(Xs)+I, C_t = cov(Xt)+I
#   Xs_align = Xs @ C_s^{-1/2} @ C_t^{1/2}
def coral(Xs, Xt, eps=1.0):
    Xs = np.asarray(Xs, float); Xt = np.asarray(Xt, float)
    d = Xs.shape[1]
    Cs = np.cov(Xs, rowvar=False) + eps*np.eye(d)
    Ct = np.cov(Xt, rowvar=False) + eps*np.eye(d)
    # whitening source lalu re-coloring ke target
    Cs_inv_sqrt = np.real(sqrtm(np.linalg.inv(Cs)))
    Ct_sqrt     = np.real(sqrtm(Ct))
    return Xs @ Cs_inv_sqrt @ Ct_sqrt

# --- Arah CIC->UNSW: align CIC(source) ke UNSW(target, tak berlabel) ---
Xc_tr_aligned = coral(Xc_tr, Xu_tr)          # source latih di-align ke distribusi target
mca = make_xgb(); mca.fit(Xc_tr_aligned, yc_tr)
coral_cic2unsw = ev(y_unsw_te, mca.predict(Xu_te))

# --- Arah UNSW->CIC: align UNSW(source) ke CIC(target) ---
Xu_tr_aligned = coral(Xu_tr, Xc_tr)
mcb = make_xgb(); mcb.fit(Xu_tr_aligned, y_unsw_tr)
coral_unsw2cic = ev(yc_te, mcb.predict(Xc_te))

results['coral'] = dict(cic2unsw=coral_cic2unsw, unsw2cic=coral_unsw2cic)
print('CORAL (unsupervised domain adaptation)')
print(f'  CIC->UNSW MCC={coral_cic2unsw["mcc"]:+.4f} F1={coral_cic2unsw["f1"]:.4f}')
print(f'  UNSW->CIC MCC={coral_unsw2cic["mcc"]:+.4f} F1={coral_unsw2cic["f1"]:.4f}')

In [ ]:
# --- Ringkasan komparatif cross-network (MCC) ---
rows = [
    dict(strategi='Baseline single-source', cic2unsw=round(results['baseline']['cic2unsw']['mcc'],4),
         unsw2cic=round(results['baseline']['unsw2cic']['mcc'],4)),
    dict(strategi='CORAL alignment', cic2unsw=round(results['coral']['cic2unsw']['mcc'],4),
         unsw2cic=round(results['coral']['unsw2cic']['mcc'],4)),
]
summ = pd.DataFrame(rows)
print('CROSS-NETWORK (MCC, uji dataset lain):')
print(summ.to_string(index=False))
print('\nJOINT TRAINING (multi-source, uji masing2 test):')
print(f"  CIC test  MCC={results['joint']['cic_test']['mcc']:.4f}")
print(f"  UNSW test MCC={results['joint']['unsw_test']['mcc']:.4f}")
print('\nInterpretasi: bandingkan CORAL vs Baseline. Naik signifikan di atas ~0 = DA berhasil.')
print('Joint = batas atas "jika kedua jaringan terlihat" (bukan generalisasi ke jaringan baru).')

In [ ]:
# --- Simpan hasil ---
meta = dict(
    deskripsi='Cross-network alignment: baseline vs joint training vs CORAL (Model A biner, z-score per dataset).',
    features=CANON,
    config=dict(model='A (9 fitur)', coral_eps=1.0, joint_ratio_cic_unsw='~3:1'),
    cross_summary=rows,
    joint=dict(cic_test_mcc=results['joint']['cic_test']['mcc'],
               unsw_test_mcc=results['joint']['unsw_test']['mcc']),
    results=results,
)
with open(OUT_JSON, 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved:', OUT_JSON)